# 8.6 实验四：从零训练避障模型

> **运行建议**：本节默认作为**训练流程教学 notebook**，建议先阅读和理解代码结构，不要求在本机完整跑完。
>
> **原因**：避障任务每一步都要采集深度图，并且每个 episode 都要 reset + takeoff + moveToPosition，AirSim 在长时间运行时更容易卡住。本机更适合做短时 sanity check，而不是完整训练。
>
> **硬件要求**：CPU 也能训练小模型，但完整实验更适合在更稳的 AirSim 环境 / GPU 机器上执行。
>
> **AirSim 配置**：使用 `settings.json`，AirSim 必须在运行。

本节在悬停训练的基础上，训练一个能够自主避障的世界模型策略。

与悬停任务的关键区别：
- 观测空间增加了**深度图**（感知障碍物距离）
- 奖励函数需要平衡**前进**和**安全**两个目标
- 训练难度更大，需要更多数据和更长的训练时间

![DreamerV3 训练闭环](figures/dreamerv3_training_loop.png)

*图 8-13：避障训练同样遵循这个闭环，但每步额外采集深度图，AirSim 交互负载更大*

![避障任务场景](figures/avoidance_scene.png)

*图 8-14：避障任务场景——无人机需要在建筑物和设施之间自主导航*

![深度图示意](figures/depth_scene.png)

*图 8-15：深度图是避障的关键感知输入——亮色表示远，暗色表示近*

In [ ]:
import sys
sys.path.append('../external-libraries')
sys.path.append('.')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import airsim
import os, time
from collections import deque

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"设备: {device}")

client = airsim.MultirotorClient()
client.confirmConnection()
DRONE = "Drone1"
FORWARD_DIR = np.array([1.0, 0.0, 0.0])
START_POS = np.array([0.0, 0.0, -8.0])

## 8.6.0 本机建议：先做短时 sanity check

如果你在一台没有独立 GPU、或者 AirSim 不够稳定的机器上学习本节，建议只做下面这两件事：

1. 跑 1~2 个短 episode，验证环境、奖励函数、深度图采集逻辑是否正常
2. 阅读完整训练代码，理解训练流程

不要一开始就完整跑 500 次迭代，因为真正的瓶颈往往不是神经网络，而是 AirSim 的长时间交互稳定性。

## 8.6.1 带深度感知的世界模型

避障任务需要感知障碍物距离。我们在状态向量中加入深度图的统计特征（左/中/右区域的平均深度和最小深度）。

In [ ]:
def get_depth_features(client, drone_id):
    """从深度图提取避障特征（6维）。"""
    responses = client.simGetImages([
        airsim.ImageRequest("0", airsim.ImageType.DepthPlanar, True)
    ], vehicle_name=drone_id)
    if responses and responses[0].width > 0:
        depth = airsim.list_to_2d_float_array(
            responses[0].image_data_float, responses[0].width, responses[0].height
        )
        depth = np.clip(depth, 0, 50)
        h, w = depth.shape
        left = depth[:, :w//3]
        center = depth[:, w//3:2*w//3]
        right = depth[:, 2*w//3:]
        return np.array([
            left.mean() / 50, center.mean() / 50, right.mean() / 50,
            left.min() / 50, center.min() / 50, right.min() / 50,
        ], dtype=np.float32)
    return np.ones(6, dtype=np.float32)

def get_full_state(client, drone_id):
    """获取完整状态：位置(3) + 速度(3) + 深度特征(6) = 12维。"""
    ms = client.getMultirotorState(vehicle_name=drone_id)
    pos = ms.kinematics_estimated.position
    vel = ms.kinematics_estimated.linear_velocity
    motion = np.array([pos.x_val, pos.y_val, pos.z_val,
                       vel.x_val, vel.y_val, vel.z_val], dtype=np.float32)
    depth_feat = get_depth_features(client, drone_id)
    return np.concatenate([motion, depth_feat])

print(f"状态维度: 12 (位置3 + 速度3 + 深度特征6)")

### 先做一个短时检查

下面这段代码不是为了训练，而是为了确认：
- AirSim 能正常返回深度图
- 状态向量维度正确（12维）
- 无人机起飞后不会立刻异常退出

如果这一步都不稳定，就不要继续完整训练。

In [ ]:
# 短时 sanity check：起飞 -> 读取状态 -> 读取深度图 -> 结束
client.reset()
client.enableApiControl(True, vehicle_name=DRONE)
client.armDisarm(True, vehicle_name=DRONE)
client.takeoffAsync(vehicle_name=DRONE).join()
client.moveToPositionAsync(*START_POS, 3, vehicle_name=DRONE).join()
time.sleep(0.5)

state = get_full_state(client, DRONE)
print(f"状态维度: {state.shape}")
print(f"位置+速度: {state[:6]}")
print(f"深度特征: {state[6:]}")
print("Sanity check OK")

## 8.6.2 数据采集与训练

流程与悬停训练类似，但状态维度更大，奖励函数更复杂。

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)
    def add(self, s, a, r, ns, d):
        self.buffer.append((s, a, r, ns, d))
    def sample(self, batch_size):
        idx = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch = [self.buffer[i] for i in idx]
        return tuple(torch.tensor(np.array([b[i] for b in batch]), dtype=torch.float32)
                     for i in range(5))
    def __len__(self):
        return len(self.buffer)

buffer = ReplayBuffer()

def collect_avoidance_episode(client, policy_fn, buffer, max_steps=300):
    client.reset()
    client.enableApiControl(True, vehicle_name=DRONE)
    client.armDisarm(True, vehicle_name=DRONE)
    client.takeoffAsync(vehicle_name=DRONE).join()
    client.moveToPositionAsync(*START_POS, 3, vehicle_name=DRONE).join()
    time.sleep(0.5)

    prev_pos = START_POS.copy()
    total_reward = 0
    collided = False

    for step in range(max_steps):
        state = get_full_state(client, DRONE)
        action = policy_fn(state)

        client.moveByRollPitchYawrateThrottleAsync(
            float(action[1])*0.3, float(action[0])*0.3,
            float(action[2])*0.5, float(action[3])*0.5+0.5,
            duration=0.1, vehicle_name=DRONE
        ).join()

        next_state = get_full_state(client, DRONE)
        pos = next_state[:3]

        collision = client.simGetCollisionInfo(vehicle_name=DRONE)
        done = collision.has_collided or abs(pos[2]) < 0.5 or pos[2] < -30

        if done:
            reward = -100.0
            collided = collision.has_collided
        else:
            forward = np.dot(pos - prev_pos, FORWARD_DIR)
            reward = forward * 2.0
            if next_state[9] < 0.1:  # center_min_depth < 5m
                reward -= 1.0

        buffer.add(state, action, reward, next_state, float(done))
        total_reward += reward
        prev_pos = pos.copy()
        if done:
            break

    forward_dist = np.dot(pos - START_POS, FORWARD_DIR)
    return total_reward, step+1, forward_dist, collided

# 采集初始数据
random_policy = lambda s: np.random.randn(4).astype(np.float32) * 0.3
print("采集初始数据...")
for ep in range(30):
    r, steps, fwd, col = collect_avoidance_episode(client, random_policy, buffer)
    if (ep+1) % 10 == 0:
        print(f"  Ep {ep+1}: reward={r:.0f}, steps={steps}, forward={fwd:.1f}m, collision={col}")
print(f"回放池: {len(buffer)} 样本")

In [ ]:
from world_model_tools import SimpleWorldModel

# 避障世界模型（状态12维，动作4维）
world_model = SimpleWorldModel(state_dim=12, action_dim=4, hidden_dim=128).to(device)
wm_opt = torch.optim.Adam(world_model.parameters(), lr=3e-4)

# 策略网络
policy = nn.Sequential(
    nn.Linear(12, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 4), nn.Tanh(),
).to(device)
pol_opt = torch.optim.Adam(policy.parameters(), lr=1e-4)

# 训练循环
wm_losses, pol_rewards = [], []
print("开始训练...")

for iteration in range(500):
    # 训练世界模型
    s, a, r, ns, d = buffer.sample(min(256, len(buffer)))
    s, a, r, ns = s.to(device), a.to(device), r.unsqueeze(1).to(device), ns.to(device)
    ps, pr = world_model(s, a)
    wm_loss = nn.MSELoss()(ps, ns) + nn.MSELoss()(pr, r)
    wm_opt.zero_grad(); wm_loss.backward(); wm_opt.step()
    wm_losses.append(wm_loss.item())

    # 在想象中训练策略
    s0, _, _, _, _ = buffer.sample(min(64, len(buffer)))
    s0 = s0.to(device)
    total_r = torch.zeros(s0.shape[0], 1, device=device)
    st = s0
    for h in range(10):
        at = policy(st)
        st, rt = world_model(st, at)
        total_r += rt * (0.99 ** h)
    pol_loss = -total_r.mean()
    pol_opt.zero_grad(); pol_loss.backward(); pol_opt.step()
    pol_rewards.append(-pol_loss.item())

    if (iteration+1) % 100 == 0:
        print(f"  Iter {iteration+1}: wm_loss={wm_loss.item():.4f}, imagined_reward={-pol_loss.item():.2f}")

print("训练完成！")

In [ ]:
# 训练曲线
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(wm_losses); axes[0].set_title('世界模型损失'); axes[0].grid(True, alpha=0.3)
axes[1].plot(pol_rewards); axes[1].set_title('想象中的累计奖励'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('avoidance_training_curves.png', dpi=150); plt.show()

## 8.6.3 保存模型并推理验证

In [ ]:
# 保存
os.makedirs('models/avoidance_checkpoint', exist_ok=True)
torch.save({
    'world_model': world_model.state_dict(),
    'policy': [p.data for p in policy.parameters()],
}, 'models/avoidance_checkpoint/model.pt')
print("模型已保存")

# 推理验证
def trained_avoidance_policy(state):
    with torch.no_grad():
        s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        return policy(s).cpu().numpy()[0]

print("\n推理验证...")
r, steps, fwd, col = collect_avoidance_episode(
    client, trained_avoidance_policy, ReplayBuffer(1), max_steps=300
)
print(f"结果: reward={r:.0f}, steps={steps}, forward={fwd:.1f}m, collision={col}")

## 8.6.4 评估指标

避障任务的关键指标：

| 指标 | 含义 | 目标 |
|------|------|------|
| 碰撞率 | 发生碰撞的 episode 比例 | 越低越好 |
| 平均前进距离 | 每个 episode 的前进距离 | 越远越好 |
| 平均存活步数 | 每个 episode 的持续时间 | 越长越好 |

理想的避障策略应该在保持低碰撞率的同时，尽可能远地前进。

In [ ]:
# 多次评估
n_eval = 5
results = []
print(f"运行 {n_eval} 次评估...")
for i in range(n_eval):
    r, steps, fwd, col = collect_avoidance_episode(
        client, trained_avoidance_policy, ReplayBuffer(1), 300
    )
    results.append({'reward': r, 'steps': steps, 'forward': fwd, 'collision': col})
    print(f"  Run {i+1}: forward={fwd:.1f}m, steps={steps}, collision={col}")

collision_rate = sum(1 for r in results if r['collision']) / n_eval
avg_forward = np.mean([r['forward'] for r in results])
avg_steps = np.mean([r['steps'] for r in results])
print(f"\n评估结果:")
print(f"  碰撞率: {collision_rate*100:.0f}%")
print(f"  平均前进距离: {avg_forward:.1f}m")
print(f"  平均存活步数: {avg_steps:.0f}")

## 8.6.5 小结

本节完成了避障任务的完整训练流程。与悬停任务相比：

| 对比 | 悬停（8.5节） | 避障（本节） |
|------|-------------|-------------|
| 状态维度 | 6（位置+速度） | 12（+深度特征） |
| 目标 | 保持不动 | 边飞边躲 |
| 奖励设计 | 单目标（靠近目标） | 多目标（前进+安全） |
| 训练难度 | 较低 | 较高 |
| AirSim 负载 | 低 | 高（每步采深度图） |

### 本机学习建议

如果本机运行 AirSim 容易卡住，请将本节视为**训练流程教学 notebook**：
- 先跑前面的 sanity check
- 阅读训练循环代码
- 不强求完整跑完所有迭代

### 离线数据备选方案

当 AirSim 长时间交互不稳定时，可以采用离线数据方案：

1. 先在 AirSim 中短时间采集一批 `(state, action, reward, next_state)` 数据并保存为 `.npz`
2. 后续训练阶段不再实时连接 AirSim，而是直接从 `.npz` 文件加载数据
3. 这样可以把“环境交互”和“模型训练”解耦，显著提高稳定性

### 进一步改进方向

1. 用 CNN 编码器直接处理深度图像，而非手工提取特征
2. 用完整的 RSSM 替代 MLP 世界模型
3. 增加数据采集轮次（采集→训练→采集→训练...迭代优化）
4. 引入课程学习（Curriculum Learning）：先在简单环境训练，逐步增加障碍物密度
5. 将训练改为离线数据模式，降低对 AirSim 稳定性的依赖